# Encoder alone

Measure performance of encoder alone without histgb

Same Transformer and preprocessing obtained into cosine_annealing_opt: width 32,
five bidirectional layers, eight heads and a regression head with two hidden layers of 16 units.

Evaluate on the same five disjoint patient folds (split seed 42) as the other comparison
notebooks, with training seeds 42, 43 and 44: **15 runs**. Each encoder trains for **160 fixed
epochs**, using 40-epoch cosine cycles with warm restarts and no internal holdout.
Preprocessing and scaling are fitted only on fold-training patients. 



In [1]:
import json
import ast
from pathlib import Path

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
encoder_definitions = json.loads(
    (PROJECT_DIR / "exploration/stacking/cosine_annealing.ipynb").read_text()
)
# Setup and function definitions only: no validation, search or final training.
for i in [2, 4, 6, 8, 10]:
    exec("".join(encoder_definitions["cells"][i]["source"]).replace(
        'print(f"Device: {DEVICE}; baseline: {baseline[-1]}")',
        'print(f"Device: {DEVICE}; final model: Transformer")'))
baseline = baseline[:2]  # Keep only the encoder preprocessing.

FIXED_EPOCHS = 160
# Import only the fixed-epoch helper, without executing its comparison loop.
fixed_source = "".join(next(c for c in encoder_definitions["cells"]
                            if c.get("id") == "cosine-fixed-cv")["source"])
fixed_function = next(f for f in ast.parse(fixed_source).body
                      if isinstance(f, ast.FunctionDef) and f.name == "fit_encoder_full")
exec(ast.get_source_segment(fixed_source, fixed_function))

# Same encoder and confirmed cosine-training parameters as post_processing_transformer.
# Evaluation uses the shared seeds 42, 43 and 44.


Device: mps; final model: Transformer


In [2]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = []
with threadpool_limits(limits=3):
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups=X["patient_id"]), 1):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline).fit(raw_train, yt)
        for seed in SEEDS:
            fitted_encoder = fit_encoder_full(raw_train, yt, prep, seed, FIXED_EPOCHS)
            _, prediction = encoder_outputs(fitted_encoder, raw_val)
            mse = mean_squared_error(yv, prediction)
            scores.append(dict(fold=fold, seed=seed, epochs=fitted_encoder["epochs"], MSE=mse))
            print(f"Fold {fold}/5, seed {seed}: MSE {mse:.4f}, RMSE {np.sqrt(mse):.4f}")
scores = pd.DataFrame(scores)
scores["RMSE"] = np.sqrt(scores["MSE"])
scores


Fold 1/5, seed 42: MSE 6.4651, RMSE 2.5427
Fold 1/5, seed 43: MSE 6.7869, RMSE 2.6052
Fold 1/5, seed 44: MSE 6.9395, RMSE 2.6343
Fold 2/5, seed 42: MSE 6.6750, RMSE 2.5836
Fold 2/5, seed 43: MSE 6.7983, RMSE 2.6074
Fold 2/5, seed 44: MSE 7.1089, RMSE 2.6663
Fold 3/5, seed 42: MSE 6.6309, RMSE 2.5751
Fold 3/5, seed 43: MSE 6.7090, RMSE 2.5902
Fold 3/5, seed 44: MSE 6.7933, RMSE 2.6064
Fold 4/5, seed 42: MSE 7.1065, RMSE 2.6658
Fold 4/5, seed 43: MSE 7.2888, RMSE 2.6998
Fold 4/5, seed 44: MSE 7.1831, RMSE 2.6801
Fold 5/5, seed 42: MSE 6.3877, RMSE 2.5274
Fold 5/5, seed 43: MSE 6.8188, RMSE 2.6113
Fold 5/5, seed 44: MSE 6.8509, RMSE 2.6174


,fold,seed,epochs,MSE,RMSE
0,1,42,160,6.465105,2.542657
1,1,43,160,6.786868,2.605162
2,1,44,160,6.939477,2.634289
3,2,42,160,6.674968,2.583596
4,2,43,160,6.798302,2.607355
5,2,44,160,7.108919,2.666256
6,3,42,160,6.630901,2.575054
7,3,43,160,6.709015,2.590177
8,3,44,160,6.793288,2.606394
9,4,42,160,7.106476,2.665798


In [3]:
summary = pd.DataFrame([dict(
    model="Transformer", MSE=scores.MSE.mean(), std=scores.MSE.std(ddof=0),
    RMSE=np.sqrt(scores.MSE.mean()),
)]).set_index("model")
summary


,MSE,std,RMSE
model,,,
Transformer,6.836178,0.246665,2.614609
